Risk Probability adalah probabilitas terjadinya gangguan rantai pasok yang diprediksi oleh model machine learning berdasarkan karakteristik transaksi logistik seperti shipping mode, lead time, scheduled delivery, dan wilayah distribusi.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = "/content/drive/MyDrive/KuliahSLRPG/"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df = pd.read_csv(path+"DataCoSupplyChainDataset.csv", sep=',', encoding='latin-1')
df.head(10)
# sumber data https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis
# https://www.kaggle.com/code/gelarerouzbahani/data-analysis-for-supply-chain

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class
5,TRANSFER,6,4,18.580000,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/19/2018 11:03,Standard Class
6,DEBIT,2,1,95.180000,288.420013,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:42,First Class
7,TRANSFER,2,1,68.430000,285.140015,Late delivery,1,73,Sporting Goods,Miami,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:21,First Class
8,CASH,3,2,133.720001,278.589996,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 10:00,Second Class
9,CASH,2,1,132.149994,275.309998,Late delivery,1,73,Sporting Goods,San Ramon,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 9:39,First Class


In [8]:
df.shape

(180519, 53)

In [36]:
df['Shipping Mode'].value_counts()

,count
Shipping Mode,
Standard Class,107752
Second Class,35216
First Class,27814
Same Day,9737


In [37]:
numeric_cols = [
    'Days for shipping (real)',
    'Days for shipment (scheduled)'
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),180519.0,3.497654,1.623722,0.0,2.0,3.0,5.0,6.0
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.0,2.0,4.0,4.0,4.0


In [38]:
region_profile = pd.DataFrame({
    'Jumlah': df['Order Region'].value_counts(),
    'Persentase (%)': round(
        df['Order Region'].value_counts(normalize=True)*100,
        2
    )
})

region_profile.head(10)

,Jumlah,Persentase (%)
Order Region,,
Central America,28341,15.70
Western Europe,27109,15.02
South America,14935,8.27
Oceania,10148,5.62
Northern Europe,9792,5.42
Southeast Asia,9539,5.28
Southern Europe,9431,5.22
Caribbean,8318,4.61
West of USA,7993,4.43


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

cols = [
    'Shipping Mode',
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Order Region'
]

data = df[cols + ['Late_delivery_risk']].copy()

for col in ['Shipping Mode','Order Region']:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

X = data.drop('Late_delivery_risk', axis=1)
y = data['Late_delivery_risk']

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1
)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97     16307
           1       0.96      1.00      0.98     19797

    accuracy                           0.97     36104
   macro avg       0.98      0.97      0.97     36104
weighted avg       0.98      0.97      0.97     36104



In [30]:
proba = model.predict_proba(X_test)

risk_df = X_test.copy()

risk_df['Risk_Probability'] = proba[:,1]

top_risk = risk_df.sort_values(
    'Risk_Probability',
    ascending=False
).head(10)

print(top_risk)

        Shipping Mode  Days for shipping (real)  \
73943               1                         1   
48082               1                         1   
112670              1                         1   
112671              1                         1   
27244               1                         1   
19774               1                         1   
48141               1                         1   
36168               1                         1   
99043               1                         1   
112518              1                         1   

        Days for shipment (scheduled)  Order Region  Risk_Probability  
73943                               0             5          0.981862  
48082                               0             5          0.981862  
112670                              0             5          0.981862  
112671                              0             4          0.981862  
27244                               0             5          0.981862  
19774 

In [31]:
risk_prob = model.predict_proba(X_test)

risk_df = X_test.copy()

risk_df["Risk_Probability"] = risk_prob[:,1]

top_risk = (
    risk_df
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
    .head(20)
)

top_risk

,Shipping Mode,Days for shipping (real),Days for shipment (scheduled),Order Region,Risk_Probability
73943,1,1,0,5,0.981862
48082,1,1,0,5,0.981862
112670,1,1,0,5,0.981862
112671,1,1,0,4,0.981862
27244,1,1,0,5,0.981862
19774,1,1,0,5,0.981862
48141,1,1,0,5,0.981862
36168,1,1,0,4,0.981862
99043,1,1,0,5,0.981862
112518,1,1,0,4,0.981862


In [32]:
!pip install groq

In [33]:
from groq import Groq
from google.colab import userdata

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

print(
    response.choices[0].message.content
)

# Executive Summary

Analisis ini bertujuan untuk mengevaluasi risiko, kerentanan, kekuatan, ketahanan, dan keberlanjutan rantai pasokan berdasarkan data transaksi yang disediakan. Hasil analisis menunjukkan bahwa rantai pasokan memiliki risiko keterlambatan yang tinggi, terutama pada mode pengiriman tertentu dan wilayah pemesanan. Oleh karena itu, perlu dilakukan mitigasi untuk mengurangi risiko dan meningkatkan ketahanan rantai pasokan.

# Supply Chain Risk Assessment

Risiko rantai pasokan dapat didefinisikan sebagai kemungkinan terjadinya gangguan atau keterlambatan dalam proses pengiriman barang. Berdasarkan data yang disediakan, dapat dilihat bahwa probabilitas keterlambatan tertinggi terjadi pada mode pengiriman dengan waktu pengiriman yang singkat (1 hari) dan wilayah pemesanan 5. Hal ini menunjukkan bahwa rantai pasokan memiliki risiko keterlambatan yang tinggi pada mode pengiriman ini. Selain itu, juga terdapat risiko keterlambatan pada mode pengiriman dengan waktu pengiriman